In [2]:
import os
new_dir = "/home/jingqi/RNALocateV3.0"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi/RNALocateV3.0'

In [3]:
import pandas as pd

### Nuclear vs Outside

In [4]:

file_path = 'Data/FIMO/a_genewise.csv'
df = pd.read_csv(file_path)

# Any location NOT in this list is considered "Outside"
nuclear_tags = {'nucleus', 'nucleoplasm', 'chromatin', 'nucleolus'}

# Define a function to classify each gene
def is_strictly_nuclear(group):
    """
    Returns True if ALL transcripts for a gene are strictly nuclear.
    Returns False if ANY transcript has a location outside the nucleus.
    """
    # Collect all localization labels for this gene across all 5 columns
    # .values.flatten() takes the 2D array of columns and makes it a 1D list
    all_locs = group[['Localization_1', 'Localization_2', 'Localization_3', 'Localization_4', 'Localization_5']].values.flatten()
    
    # Filter out 'nan' (empty) values and clean the strings
    cleaned_locs = [str(x).strip() for x in all_locs if str(x) != 'nan']
    
    # Check: Are ALL locations in our nuclear_tags set?
    return all(loc in nuclear_tags for loc in cleaned_locs)

# Group the data by Gene and apply the classification
strictly_nuclear_genes = []
non_nuclear_genes = []

# Iterate through each gene group
for gene, group in df.groupby('Gene'):
    if is_strictly_nuclear(group):
        strictly_nuclear_genes.append(gene)
    else:
        non_nuclear_genes.append(gene)

# Create the two DataFrames
df_strictly_nuclear = df[df['Gene'].isin(strictly_nuclear_genes)]
df_with_outside = df[df['Gene'].isin(non_nuclear_genes)]

# Save the results to CSV files
df_strictly_nuclear.to_csv('Data/Validation/genes_strictly_nuclear.csv', index=False)
df_with_outside.to_csv('Data/Validation/genes_notonly_nuclear.csv', index=False)

print(f"Processing Complete.")
print(f"Total Genes Processed: {len(df['Gene'].unique())}")
print(f" - Strictly Nuclear Genes: {len(strictly_nuclear_genes)} (Saved to 'genes_strictly_nuclear.csv')")
print(f" - Genes with Outside Transcripts: {len(non_nuclear_genes)} (Saved to 'genes_with_non_nuclear.csv')")

Processing Complete.
Total Genes Processed: 433
 - Strictly Nuclear Genes: 293 (Saved to 'genes_strictly_nuclear.csv')
 - Genes with Outside Transcripts: 140 (Saved to 'genes_with_non_nuclear.csv')


### Extracellular vs Intracellylar

In [8]:
file_path = 'Data/Validation/genes_notonly_nuclear.csv'
df = pd.read_csv(file_path)

# 2. Define tags for Extracellular locations
extracellular_tags = 'extracellular'

# 3. Define a function to check if a gene has ANY extracellular transcript
def is_extracellular_gene(group):
    # Flatten all localizations for this gene
    all_locs = group[['Localization_1', 'Localization_2', 'Localization_3', 'Localization_4', 'Localization_5']].values.flatten()

    for loc in all_locs:
            if str(loc).strip() == extracellular_tags:
                return True
    return False

# Classify the genes
genes_extracellular = []
genes_intracellular = []

for gene, group in df.groupby('Gene'):
    if is_extracellular_gene(group):
        genes_extracellular.append(gene)
    else:
        # If it's in this file, it's already "non-nuclear".
        # If it's not extracellular, it must be Intracellular (Cytoplasm/Mito/etc)
        genes_intracellular.append(gene)

# Create DataFrames
df_extracellular = df[df['Gene'].isin(genes_extracellular)]
df_intracellular = df[df['Gene'].isin(genes_intracellular)]

#  Save to CSV
df_extracellular.to_csv('Data/Validation/genes_extracellular.csv', index=False)
df_intracellular.to_csv('Data/Validation/genes_intracellular_non_nuclear.csv', index=False)
print(f"Processing Complete.")
print(f"Total Non-Nuclear Genes: {len(df['Gene'].unique())}")
print(f" - Extracellular/Secreted Genes: {len(genes_extracellular)} (Saved to 'genes_extracellular.csv')")
print(f" - Intracellular Non-Nuclear Genes: {len(genes_intracellular)} (Saved to 'genes_intracellular_non_nuclear.csv')")

Processing Complete.
Total Non-Nuclear Genes: 140
 - Extracellular/Secreted Genes: 100 (Saved to 'genes_extracellular.csv')
 - Intracellular Non-Nuclear Genes: 40 (Saved to 'genes_intracellular_non_nuclear.csv')


### extract the gene names

In [10]:
df_extra = pd.read_csv('Data/Validation/genes_extracellular.csv')
df_intra = pd.read_csv('Data/Validation/genes_intracellular_non_nuclear.csv')

# Extract unique gene names
genes_extra_list = df_extra['Gene'].unique()
genes_intra_list = df_intra['Gene'].unique()

with open('Data/Validation/gene_names_extracellular.txt', 'w') as f:
    for gene in genes_extra_list:
        f.write(f"{gene}\n")

with open('Data/Validation/gene_names_intracellular_non_nuclear.txt', 'w') as f:
    for gene in genes_intra_list:
        f.write(f"{gene}\n")

print(f"Extraction Complete.")

Extraction Complete.
